In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

# Boruta Ensemble Validation

This notebook runs **every strategy** on a single ticker, generates all OR-logic ensemble combinations,
ranks by IS Sharpe, then validates with **Boruta** (shuffle OOS returns to test statistical significance).

**Workflow:**
1. Run 7 individual strategies on train data
2. Generate all 2..N ensemble combinations (OR logic)
3. Rank by IS Sharpe
4. Boruta-validate top 25 ensembles on OOS data
5. Visualize and export results

In [ ]:
# !pip install vectorbt yfinance TA-Lib numpy pandas matplotlib seaborn scipy

## Imports

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
import warnings
import json
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('All imports loaded successfully.')
from lib.data_manager import download, download_multi, setup_colab_secrets

## Configuration

In [ ]:
TICKER = "QQQ"
START_DATE = "2018-01-01"
TRAIN_RATIO = 0.60
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
MIN_TRADES = 10
N_SHUFFLES = 500       # Boruta shuffle count
BORUTA_THRESHOLD = 0.80  # 80% significance

print(f"Ticker: {TICKER}")
print(f"Start Date: {START_DATE}")
print(f"Train Ratio: {TRAIN_RATIO}")
print(f"Boruta Shuffles: {N_SHUFFLES}")
print(f"Boruta Threshold: {BORUTA_THRESHOLD}")

## Download Data

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
raw = download(TICKER, START_DATE)

close = raw['Close'].astype(float)
high = raw['High'].astype(float)
low = raw['Low'].astype(float)
open_ = raw['Open'].astype(float)
volume = raw['Volume'].astype(float)

print(f"Downloaded {len(close)} bars for {TICKER}")
print(f"Date range: {close.index[0].date()} to {close.index[-1].date()}")
print(f"Close price range: ${close.min():.2f} - ${close.max():.2f}")

## Train / Test Split

In [ ]:
split_idx = int(len(close) * TRAIN_RATIO)

# Full data references (for signal generation on full data)
close_full = close
high_full = high
low_full = low

# Train data
close_train = close.iloc[:split_idx]
high_train = high.iloc[:split_idx]
low_train = low.iloc[:split_idx]

# Validation data
close_val = close.iloc[split_idx:]
high_val = high.iloc[split_idx:]
low_val = low.iloc[split_idx:]

print(f"Train: {close_train.index[0].date()} to {close_train.index[-1].date()} ({len(close_train)} bars)")
print(f"Val:   {close_val.index[0].date()} to {close_val.index[-1].date()} ({len(close_val)} bars)")
print(f"Split index: {split_idx}")

## Strategy Signal Generators

Each function generates signals on the **full dataset**, then we slice to train/val for backtesting.
All strategies use 1-bar execution delay.

In [ ]:
def strat_macd(close, high=None, low=None):
    """MACD Crossover (fast=12, slow=26, signal=9)"""
    macd, signal_line, _ = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
    entries_raw = (macd > signal_line) & (macd.shift(1) <= signal_line.shift(1))
    exits_raw = (macd < signal_line) & (macd.shift(1) >= signal_line.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'MACD', 'entries': entries, 'exits': exits}


def strat_rsi(close, high=None, low=None):
    """RSI Mean Reversion (period=14, oversold=30, overbought=70)"""
    rsi = talib.RSI(close, timeperiod=14)
    entries_raw = (rsi > 30) & (rsi.shift(1) <= 30)
    exits_raw = (rsi > 70) & (rsi.shift(1) <= 70)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'RSI', 'entries': entries, 'exits': exits}


def strat_ema(close, high=None, low=None):
    """EMA Crossover (fast=12, slow=26, trend=50)"""
    fast_ema = talib.EMA(close, timeperiod=12)
    slow_ema = talib.EMA(close, timeperiod=26)
    trend_sma = talib.SMA(close, timeperiod=50)
    entries_raw = (fast_ema > slow_ema) & (fast_ema.shift(1) <= slow_ema.shift(1)) & (close > trend_sma)
    exits_raw = (fast_ema < slow_ema) & (fast_ema.shift(1) >= slow_ema.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'EMA', 'entries': entries, 'exits': exits}


def strat_donchian(close, high=None, low=None):
    """Donchian Breakout (entry=20, exit=10, filter=50)"""
    upper = talib.MAX(high, timeperiod=20).shift(1)
    lower = talib.MIN(low, timeperiod=10).shift(1)
    trend_filter = talib.SMA(close, timeperiod=50).shift(1)
    entries_raw = (close > upper) & (close > trend_filter)
    exits_raw = (close < lower)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'Donchian', 'entries': entries, 'exits': exits}


def strat_bbands(close, high=None, low=None):
    """Bollinger Band Mean Reversion (period=20, std=2.0)"""
    upper, mid, lower_bb = talib.BBANDS(close, timeperiod=20, nbdevup=2.0, nbdevdn=2.0)
    entries_raw = (close < lower_bb) & (close.shift(1) >= lower_bb.shift(1))
    exits_raw = (close > mid) & (close.shift(1) <= mid.shift(1))
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'BBands', 'entries': entries, 'exits': exits}


def strat_atr_breakout(close, high=None, low=None):
    """ATR Volatility Breakout (atr=14, sma=20, mult=2.0)"""
    atr = talib.ATR(high, low, close, timeperiod=14)
    sma = talib.SMA(close, timeperiod=20)
    atr_mult = 2.0
    entries_raw = close > (sma + atr_mult * atr)
    exits_raw = close < (sma - 1.0 * atr)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'ATR_Breakout', 'entries': entries, 'exits': exits}


def strat_stochastic(close, high=None, low=None):
    """Stochastic Oscillator (k=14, d=3, oversold=20, overbought=80)"""
    slowk, slowd = talib.STOCH(
        high, low, close,
        fastk_period=14, slowk_period=3, slowk_matype=0,
        slowd_period=3, slowd_matype=0
    )
    entries_raw = (slowk > slowd) & (slowk.shift(1) <= slowd.shift(1)) & (slowk < 20)
    exits_raw = (slowk < slowd) & (slowk.shift(1) >= slowd.shift(1)) & (slowk > 80)
    entries = entries_raw.shift(1).fillna(False).astype(bool)
    exits = exits_raw.shift(1).fillna(False).astype(bool)
    return {'name': 'Stochastic', 'entries': entries, 'exits': exits}


# Generate all strategy signals on full data
strategy_funcs = [strat_macd, strat_rsi, strat_ema, strat_donchian,
                  strat_bbands, strat_atr_breakout, strat_stochastic]

strategies = []
for func in strategy_funcs:
    s = func(close_full, high=high_full, low=low_full)
    strategies.append(s)
    n_entries = s['entries'].sum()
    n_exits = s['exits'].sum()
    print(f"{s['name']:20s}  entries={n_entries:4d}  exits={n_exits:4d}")

print(f"\nTotal strategies: {len(strategies)}")

## Individual Strategy Backtest (IS + OOS)

In [ ]:
def run_backtest(entries, exits, price, init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE):
    """Run vectorbt backtest and return portfolio object."""
    pf = vbt.Portfolio.from_signals(
        price,
        entries=entries,
        exits=exits,
        init_cash=init_cash,
        fees=fees,
        slippage=slippage,
        freq='D'
    )
    return pf


def get_metrics(pf):
    """Extract key metrics from a portfolio."""
    try:
        sharpe = pf.sharpe_ratio()
    except Exception:
        sharpe = np.nan
    try:
        total_return = pf.total_return()
    except Exception:
        total_return = np.nan
    try:
        max_dd = pf.max_drawdown()
    except Exception:
        max_dd = np.nan
    try:
        n_trades = pf.trades.count()
    except Exception:
        n_trades = 0
    return {
        'sharpe': sharpe,
        'total_return': total_return,
        'max_dd': max_dd,
        'n_trades': n_trades
    }


# Run backtests on train and val for each individual strategy
individual_results = []

for s in strategies:
    # Train (IS)
    ent_train = s['entries'].iloc[:split_idx]
    ext_train = s['exits'].iloc[:split_idx]
    pf_train = run_backtest(ent_train, ext_train, close_train)
    m_train = get_metrics(pf_train)

    # Val (OOS)
    ent_val = s['entries'].iloc[split_idx:]
    ext_val = s['exits'].iloc[split_idx:]
    pf_val = run_backtest(ent_val, ext_val, close_val)
    m_val = get_metrics(pf_val)

    individual_results.append({
        'name': s['name'],
        'is_sharpe': m_train['sharpe'],
        'is_return': m_train['total_return'],
        'is_maxdd': m_train['max_dd'],
        'is_trades': m_train['n_trades'],
        'oos_sharpe': m_val['sharpe'],
        'oos_return': m_val['total_return'],
        'oos_maxdd': m_val['max_dd'],
        'oos_trades': m_val['n_trades'],
    })

# Display
df_individual = pd.DataFrame(individual_results)
print("Individual Strategy Results")
print("=" * 100)
fmt_cols = {
    'is_sharpe': '{:.3f}', 'is_return': '{:.2%}', 'is_maxdd': '{:.2%}',
    'oos_sharpe': '{:.3f}', 'oos_return': '{:.2%}', 'oos_maxdd': '{:.2%}'
}
display_df = df_individual.copy()
for col, fmt in fmt_cols.items():
    display_df[col] = df_individual[col].apply(lambda x: fmt.format(x) if pd.notna(x) else 'N/A')
print(display_df.to_string(index=False))

## Generate Ensemble Combinations (OR Logic)

We combine every possible subset (2 or more strategies) using OR logic for entries and exits,
then backtest each on the train set.

In [ ]:
strategy_names = [s['name'] for s in strategies]

all_combos = []
for r in range(2, len(strategies) + 1):
    for combo in combinations(range(len(strategies)), r):
        all_combos.append(combo)

print(f"Total ensemble combinations: {len(all_combos)}")
print(f"(from pairs to full {len(strategies)}-strategy ensemble)")

# Backtest each combination on train data
ensemble_results = []

for combo in all_combos:
    combo_name = ' + '.join([strategies[i]['name'] for i in combo])

    # OR logic: combine entries and exits
    ent_combined = strategies[combo[0]]['entries'].iloc[:split_idx].copy()
    ext_combined = strategies[combo[0]]['exits'].iloc[:split_idx].copy()
    for i in combo[1:]:
        ent_combined = ent_combined | strategies[i]['entries'].iloc[:split_idx]
        ext_combined = ext_combined | strategies[i]['exits'].iloc[:split_idx]

    # Backtest on train
    pf_train = run_backtest(ent_combined, ext_combined, close_train)
    m_train = get_metrics(pf_train)

    if m_train['n_trades'] < MIN_TRADES:
        continue

    ensemble_results.append({
        'combo': combo,
        'name': combo_name,
        'n_strategies': len(combo),
        'is_sharpe': m_train['sharpe'],
        'is_return': m_train['total_return'],
        'is_maxdd': m_train['max_dd'],
        'is_trades': m_train['n_trades'],
    })

# Sort by IS Sharpe descending
ensemble_results.sort(key=lambda x: x['is_sharpe'] if pd.notna(x['is_sharpe']) else -999, reverse=True)

print(f"\nValid ensembles (>= {MIN_TRADES} trades): {len(ensemble_results)}")
print(f"\nTop 25 Ensembles by IS Sharpe:")
print("=" * 100)
print(f"{'Rank':<5} {'Ensemble':<55} {'IS Sharpe':>10} {'IS Return':>10} {'IS MaxDD':>10} {'Trades':>7}")
print("-" * 100)
for i, er in enumerate(ensemble_results[:25]):
    sharpe_str = f"{er['is_sharpe']:.3f}" if pd.notna(er['is_sharpe']) else 'N/A'
    ret_str = f"{er['is_return']:.2%}" if pd.notna(er['is_return']) else 'N/A'
    dd_str = f"{er['is_maxdd']:.2%}" if pd.notna(er['is_maxdd']) else 'N/A'
    print(f"{i+1:<5} {er['name']:<55} {sharpe_str:>10} {ret_str:>10} {dd_str:>10} {er['is_trades']:>7}")

#
#
 
B
o
r
u
t
a
 
V
a
l
i
d
a
t
i
o
n
 
(
C
o
r
r
e
c
t
e
d
)


F
o
r
 
e
a
c
h
 
t
o
p
 
e
n
s
e
m
b
l
e
,
 
w
e
 
c
o
m
p
a
r
e
 
t
h
e
 
r
e
a
l
 
O
O
S
 
S
h
a
r
p
e
 
a
g
a
i
n
s
t
 
s
h
a
d
o
w
 
S
h
a
r
p
e
s
.


*
*
M
e
t
h
o
d
:
*
*
 
S
h
u
f
f
l
e
 
t
h
e
 
*
*
a
s
s
e
t
'
s
*
*
 
O
O
S
 
r
e
t
u
r
n
s
 
(
n
o
t
 
t
h
e
 
p
o
r
t
f
o
l
i
o
 
r
e
t
u
r
n
s
)
,
 
t
h
e
n
 
r
e
-
a
p
p
l
y

t
h
e
 
s
a
m
e
 
p
o
s
i
t
i
o
n
 
i
n
d
i
c
a
t
o
r
 
t
o
 
t
h
e
 
s
h
u
f
f
l
e
d
 
r
e
t
u
r
n
s
.
 
T
h
i
s
 
t
e
s
t
s
 
w
h
e
t
h
e
r
 
t
h
e
 
s
t
r
a
t
e
g
y
'
s

*
t
i
m
i
n
g
*
 
o
f
 
e
n
t
r
i
e
s
/
e
x
i
t
s
 
a
d
d
s
 
v
a
l
u
e
 
b
e
y
o
n
d
 
r
a
n
d
o
m
 
c
h
a
n
c
e
.


*
*
M
u
l
t
i
p
l
e
-
t
e
s
t
i
n
g
 
c
o
r
r
e
c
t
i
o
n
:
*
*
 
B
e
n
j
a
m
i
n
i
-
H
o
c
h
b
e
r
g
 
F
D
R
 
a
t
 
5
%
 
c
o
n
t
r
o
l
s
 
t
h
e
 
f
a
l
s
e
 
d
i
s
c
o
v
e
r
y

r
a
t
e
 
a
c
r
o
s
s
 
a
l
l
 
2
5
 
s
i
m
u
l
t
a
n
e
o
u
s
 
t
e
s
t
s
.

In [ ]:
d
e
f
 
g
e
t
_
p
o
s
i
t
i
o
n
_
s
e
r
i
e
s
(
e
n
t
r
i
e
s
,
 
e
x
i
t
s
,
 
p
r
i
c
e
)
:

 
 
 
 
"
"
"
C
o
n
v
e
r
t
 
e
n
t
r
y
/
e
x
i
t
 
s
i
g
n
a
l
s
 
i
n
t
o
 
a
 
b
i
n
a
r
y
 
p
o
s
i
t
i
o
n
 
s
e
r
i
e
s
 
(
1
=
i
n
,
 
0
=
o
u
t
)
.

 
 
 
 
U
s
e
s
 
v
e
c
t
o
r
b
t
 
t
o
 
h
a
n
d
l
e
 
s
i
g
n
a
l
 
l
o
g
i
c
 
c
o
r
r
e
c
t
l
y
 
(
n
o
 
d
o
u
b
l
e
 
e
n
t
r
i
e
s
,
 
e
t
c
.
)
.
"
"
"

 
 
 
 
p
f
 
=
 
r
u
n
_
b
a
c
k
t
e
s
t
(
e
n
t
r
i
e
s
,
 
e
x
i
t
s
,
 
p
r
i
c
e
)

 
 
 
 
#
 
P
o
s
i
t
i
o
n
 
a
s
 
f
r
a
c
t
i
o
n
 
o
f
 
p
o
r
t
f
o
l
i
o
 
i
n
 
t
h
e
 
a
s
s
e
t
;
 
c
o
n
v
e
r
t
 
t
o
 
b
i
n
a
r
y

 
 
 
 
p
o
s
 
=
 
p
f
.
a
s
s
e
t
_
v
a
l
u
e
(
)
 
/
 
p
f
.
v
a
l
u
e
(
)

 
 
 
 
p
o
s
i
t
i
o
n
 
=
 
(
p
o
s
 
>
 
0
.
0
1
)
.
a
s
t
y
p
e
(
i
n
t
)
 
 
#
 
1
 
i
f
 
i
n
 
p
o
s
i
t
i
o
n
,
 
0
 
i
f
 
f
l
a
t

 
 
 
 
r
e
t
u
r
n
 
p
o
s
i
t
i
o
n



d
e
f
 
b
o
r
u
t
a
_
v
a
l
i
d
a
t
e
(
p
o
s
i
t
i
o
n
_
o
o
s
,
 
a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
,
 
n
_
s
h
u
f
f
l
e
s
=
5
0
0
)
:

 
 
 
 
"
"
"

 
 
 
 
C
o
r
r
e
c
t
e
d
 
B
o
r
u
t
a
 
v
a
l
i
d
a
t
i
o
n
.


 
 
 
 
S
h
u
f
f
l
e
s
 
t
h
e
 
A
S
S
E
T
 
r
e
t
u
r
n
s
 
w
h
i
l
e
 
k
e
e
p
i
n
g
 
t
h
e
 
p
o
s
i
t
i
o
n
 
i
n
d
i
c
a
t
o
r
 
f
i
x
e
d
.

 
 
 
 
T
h
i
s
 
t
e
s
t
s
 
w
h
e
t
h
e
r
 
t
h
e
 
s
t
r
a
t
e
g
y
'
s
 
t
i
m
i
n
g
 
a
d
d
s
 
v
a
l
u
e
.


 
 
 
 
P
a
r
a
m
e
t
e
r
s

 
 
 
 
-
-
-
-
-
-
-
-
-
-

 
 
 
 
p
o
s
i
t
i
o
n
_
o
o
s
 
:
 
p
d
.
S
e
r
i
e
s

 
 
 
 
 
 
 
 
B
i
n
a
r
y
 
p
o
s
i
t
i
o
n
 
s
e
r
i
e
s
 
(
1
=
i
n
 
m
a
r
k
e
t
,
 
0
=
f
l
a
t
)
 
o
n
 
O
O
S
 
p
e
r
i
o
d
.

 
 
 
 
a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
 
:
 
p
d
.
S
e
r
i
e
s

 
 
 
 
 
 
 
 
T
h
e
 
a
s
s
e
t
'
s
 
d
a
i
l
y
 
r
e
t
u
r
n
s
 
o
n
 
O
O
S
 
p
e
r
i
o
d
.

 
 
 
 
n
_
s
h
u
f
f
l
e
s
 
:
 
i
n
t

 
 
 
 
 
 
 
 
N
u
m
b
e
r
 
o
f
 
p
e
r
m
u
t
a
t
i
o
n
s
.


 
 
 
 
R
e
t
u
r
n
s

 
 
 
 
-
-
-
-
-
-
-

 
 
 
 
b
o
r
u
t
a
_
s
c
o
r
e
 
:
 
f
l
o
a
t

 
 
 
 
 
 
 
 
F
r
a
c
t
i
o
n
 
o
f
 
s
h
u
f
f
l
e
s
 
w
h
e
r
e
 
r
e
a
l
 
S
h
a
r
p
e
 
>
 
s
h
a
d
o
w
 
S
h
a
r
p
e
.

 
 
 
 
r
e
a
l
_
s
h
a
r
p
e
 
:
 
f
l
o
a
t

 
 
 
 
 
 
 
 
T
h
e
 
a
c
t
u
a
l
 
s
t
r
a
t
e
g
y
 
S
h
a
r
p
e
 
o
n
 
O
O
S
.

 
 
 
 
m
e
d
i
a
n
_
s
h
a
d
o
w
 
:
 
f
l
o
a
t

 
 
 
 
 
 
 
 
M
e
d
i
a
n
 
o
f
 
s
h
a
d
o
w
 
S
h
a
r
p
e
s
.

 
 
 
 
"
"
"

 
 
 
 
#
 
A
l
i
g
n
 
i
n
d
i
c
e
s

 
 
 
 
c
o
m
m
o
n
_
i
d
x
 
=
 
p
o
s
i
t
i
o
n
_
o
o
s
.
i
n
d
e
x
.
i
n
t
e
r
s
e
c
t
i
o
n
(
a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
.
i
n
d
e
x
)

 
 
 
 
p
o
s
 
=
 
p
o
s
i
t
i
o
n
_
o
o
s
.
l
o
c
[
c
o
m
m
o
n
_
i
d
x
]
.
v
a
l
u
e
s

 
 
 
 
a
s
s
e
t
_
r
e
t
 
=
 
a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
.
l
o
c
[
c
o
m
m
o
n
_
i
d
x
]
.
v
a
l
u
e
s


 
 
 
 
#
 
S
t
r
a
t
e
g
y
 
r
e
t
u
r
n
s
 
=
 
a
s
s
e
t
 
r
e
t
u
r
n
 
w
h
e
n
 
i
n
 
p
o
s
i
t
i
o
n
,
 
0
 
w
h
e
n
 
f
l
a
t

 
 
 
 
s
t
r
a
t
_
r
e
t
 
=
 
p
o
s
 
*
 
a
s
s
e
t
_
r
e
t


 
 
 
 
#
 
R
e
m
o
v
e
 
l
e
a
d
i
n
g
 
N
a
N
s

 
 
 
 
m
a
s
k
 
=
 
~
n
p
.
i
s
n
a
n
(
s
t
r
a
t
_
r
e
t
)
 
&
 
~
n
p
.
i
s
n
a
n
(
a
s
s
e
t
_
r
e
t
)

 
 
 
 
s
t
r
a
t
_
r
e
t
 
=
 
s
t
r
a
t
_
r
e
t
[
m
a
s
k
]

 
 
 
 
a
s
s
e
t
_
r
e
t
_
c
l
e
a
n
 
=
 
a
s
s
e
t
_
r
e
t
[
m
a
s
k
]

 
 
 
 
p
o
s
_
c
l
e
a
n
 
=
 
p
o
s
[
m
a
s
k
]


 
 
 
 
i
f
 
l
e
n
(
s
t
r
a
t
_
r
e
t
)
 
<
 
2
0
 
o
r
 
n
p
.
s
t
d
(
s
t
r
a
t
_
r
e
t
)
 
=
=
 
0
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
0
.
0
,
 
n
p
.
n
a
n
,
 
n
p
.
n
a
n


 
 
 
 
r
e
a
l
_
s
h
a
r
p
e
 
=
 
n
p
.
m
e
a
n
(
s
t
r
a
t
_
r
e
t
)
 
/
 
n
p
.
s
t
d
(
s
t
r
a
t
_
r
e
t
)
 
*
 
n
p
.
s
q
r
t
(
2
5
2
)


 
 
 
 
#
 
S
h
u
f
f
l
e
 
a
s
s
e
t
 
r
e
t
u
r
n
s
,
 
r
e
-
a
p
p
l
y
 
s
a
m
e
 
p
o
s
i
t
i
o
n
 
i
n
d
i
c
a
t
o
r

 
 
 
 
s
h
a
d
o
w
_
s
h
a
r
p
e
s
 
=
 
[
]

 
 
 
 
f
o
r
 
_
 
i
n
 
r
a
n
g
e
(
n
_
s
h
u
f
f
l
e
s
)
:

 
 
 
 
 
 
 
 
s
h
u
f
f
l
e
d
_
a
s
s
e
t
 
=
 
n
p
.
r
a
n
d
o
m
.
p
e
r
m
u
t
a
t
i
o
n
(
a
s
s
e
t
_
r
e
t
_
c
l
e
a
n
)

 
 
 
 
 
 
 
 
s
h
a
d
o
w
_
r
e
t
 
=
 
p
o
s
_
c
l
e
a
n
 
*
 
s
h
u
f
f
l
e
d
_
a
s
s
e
t

 
 
 
 
 
 
 
 
s
_
s
t
d
 
=
 
n
p
.
s
t
d
(
s
h
a
d
o
w
_
r
e
t
)

 
 
 
 
 
 
 
 
i
f
 
s
_
s
t
d
 
=
=
 
0
:

 
 
 
 
 
 
 
 
 
 
 
 
s
h
a
d
o
w
_
s
h
a
r
p
e
s
.
a
p
p
e
n
d
(
0
.
0
)

 
 
 
 
 
 
 
 
e
l
s
e
:

 
 
 
 
 
 
 
 
 
 
 
 
s
h
a
d
o
w
_
s
h
a
r
p
e
s
.
a
p
p
e
n
d
(
n
p
.
m
e
a
n
(
s
h
a
d
o
w
_
r
e
t
)
 
/
 
s
_
s
t
d
 
*
 
n
p
.
s
q
r
t
(
2
5
2
)
)


 
 
 
 
b
o
r
u
t
a
_
s
c
o
r
e
 
=
 
n
p
.
m
e
a
n
(
r
e
a
l
_
s
h
a
r
p
e
 
>
 
n
p
.
a
r
r
a
y
(
s
h
a
d
o
w
_
s
h
a
r
p
e
s
)
)

 
 
 
 
r
e
t
u
r
n
 
b
o
r
u
t
a
_
s
c
o
r
e
,
 
r
e
a
l
_
s
h
a
r
p
e
,
 
n
p
.
m
e
d
i
a
n
(
s
h
a
d
o
w
_
s
h
a
r
p
e
s
)



d
e
f
 
b
e
n
j
a
m
i
n
i
_
h
o
c
h
b
e
r
g
(
p
_
v
a
l
u
e
s
,
 
f
d
r
=
0
.
0
5
)
:

 
 
 
 
"
"
"

 
 
 
 
B
e
n
j
a
m
i
n
i
-
H
o
c
h
b
e
r
g
 
F
D
R
 
c
o
r
r
e
c
t
i
o
n
.


 
 
 
 
P
a
r
a
m
e
t
e
r
s

 
 
 
 
-
-
-
-
-
-
-
-
-
-

 
 
 
 
p
_
v
a
l
u
e
s
 
:
 
l
i
s
t
 
o
f
 
f
l
o
a
t

 
 
 
 
f
d
r
 
:
 
f
l
o
a
t

 
 
 
 
 
 
 
 
F
a
l
s
e
 
d
i
s
c
o
v
e
r
y
 
r
a
t
e
 
(
d
e
f
a
u
l
t
 
5
%
)
.


 
 
 
 
R
e
t
u
r
n
s

 
 
 
 
-
-
-
-
-
-
-

 
 
 
 
r
e
j
e
c
t
e
d
 
:
 
l
i
s
t
 
o
f
 
b
o
o
l

 
 
 
 
 
 
 
 
T
r
u
e
 
i
f
 
t
h
e
 
n
u
l
l
 
h
y
p
o
t
h
e
s
i
s
 
i
s
 
r
e
j
e
c
t
e
d
 
(
e
n
s
e
m
b
l
e
 
i
s
 
s
i
g
n
i
f
i
c
a
n
t
)
.

 
 
 
 
a
d
j
u
s
t
e
d
_
p
v
a
l
s
 
:
 
l
i
s
t
 
o
f
 
f
l
o
a
t

 
 
 
 
 
 
 
 
F
D
R
-
a
d
j
u
s
t
e
d
 
p
-
v
a
l
u
e
s
.

 
 
 
 
"
"
"

 
 
 
 
n
 
=
 
l
e
n
(
p
_
v
a
l
u
e
s
)

 
 
 
 
i
n
d
e
x
e
d
 
=
 
s
o
r
t
e
d
(
e
n
u
m
e
r
a
t
e
(
p
_
v
a
l
u
e
s
)
,
 
k
e
y
=
l
a
m
b
d
a
 
x
:
 
x
[
1
]
)


 
 
 
 
a
d
j
u
s
t
e
d
 
=
 
[
0
.
0
]
 
*
 
n

 
 
 
 
r
e
j
e
c
t
e
d
 
=
 
[
F
a
l
s
e
]
 
*
 
n


 
 
 
 
#
 
C
o
m
p
u
t
e
 
a
d
j
u
s
t
e
d
 
p
-
v
a
l
u
e
s
 
(
s
t
e
p
-
u
p
)

 
 
 
 
p
r
e
v
_
a
d
j
 
=
 
1
.
0

 
 
 
 
f
o
r
 
r
a
n
k
_
f
r
o
m
_
e
n
d
,
 
(
o
r
i
g
_
i
d
x
,
 
p
v
a
l
)
 
i
n
 
e
n
u
m
e
r
a
t
e
(
r
e
v
e
r
s
e
d
(
i
n
d
e
x
e
d
)
)
:

 
 
 
 
 
 
 
 
r
a
n
k
 
=
 
n
 
-
 
r
a
n
k
_
f
r
o
m
_
e
n
d
 
 
#
 
1
-
i
n
d
e
x
e
d
 
r
a
n
k

 
 
 
 
 
 
 
 
a
d
j
_
p
 
=
 
m
i
n
(
p
r
e
v
_
a
d
j
,
 
p
v
a
l
 
*
 
n
 
/
 
r
a
n
k
)

 
 
 
 
 
 
 
 
a
d
j
_
p
 
=
 
m
i
n
(
a
d
j
_
p
,
 
1
.
0
)

 
 
 
 
 
 
 
 
a
d
j
u
s
t
e
d
[
o
r
i
g
_
i
d
x
]
 
=
 
a
d
j
_
p

 
 
 
 
 
 
 
 
p
r
e
v
_
a
d
j
 
=
 
a
d
j
_
p


 
 
 
 
f
o
r
 
i
 
i
n
 
r
a
n
g
e
(
n
)
:

 
 
 
 
 
 
 
 
r
e
j
e
c
t
e
d
[
i
]
 
=
 
a
d
j
u
s
t
e
d
[
i
]
 
<
=
 
f
d
r


 
 
 
 
r
e
t
u
r
n
 
r
e
j
e
c
t
e
d
,
 
a
d
j
u
s
t
e
d



#
 
-
-
-
-
 
R
u
n
 
B
o
r
u
t
a
 
o
n
 
t
o
p
 
2
5
 
e
n
s
e
m
b
l
e
s
 
-
-
-
-


#
 
C
o
m
p
u
t
e
 
a
s
s
e
t
 
O
O
S
 
r
e
t
u
r
n
s
 
o
n
c
e

a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
 
=
 
c
l
o
s
e
_
v
a
l
.
p
c
t
_
c
h
a
n
g
e
(
)
.
d
r
o
p
n
a
(
)


t
o
p
_
n
 
=
 
m
i
n
(
2
5
,
 
l
e
n
(
e
n
s
e
m
b
l
e
_
r
e
s
u
l
t
s
)
)

t
o
p
_
e
n
s
e
m
b
l
e
s
 
=
 
e
n
s
e
m
b
l
e
_
r
e
s
u
l
t
s
[
:
t
o
p
_
n
]


p
r
i
n
t
(
f
"
R
u
n
n
i
n
g
 
B
o
r
u
t
a
 
v
a
l
i
d
a
t
i
o
n
 
o
n
 
t
o
p
 
{
t
o
p
_
n
}
 
e
n
s
e
m
b
l
e
s
 
(
{
N
_
S
H
U
F
F
L
E
S
}
 
s
h
u
f
f
l
e
s
 
e
a
c
h
)
.
.
.
"
)

p
r
i
n
t
(
f
"
M
e
t
h
o
d
:
 
s
h
u
f
f
l
e
 
a
s
s
e
t
 
r
e
t
u
r
n
s
,
 
r
e
-
a
p
p
l
y
 
p
o
s
i
t
i
o
n
 
i
n
d
i
c
a
t
o
r
"
)

p
r
i
n
t
(
f
"
M
u
l
t
i
p
l
e
-
t
e
s
t
i
n
g
 
c
o
r
r
e
c
t
i
o
n
:
 
B
e
n
j
a
m
i
n
i
-
H
o
c
h
b
e
r
g
 
F
D
R
 
@
 
5
%
"
)

p
r
i
n
t
(
)


b
o
r
u
t
a
_
r
e
s
u
l
t
s
 
=
 
[
]


f
o
r
 
i
d
x
,
 
e
r
 
i
n
 
e
n
u
m
e
r
a
t
e
(
t
o
p
_
e
n
s
e
m
b
l
e
s
)
:

 
 
 
 
c
o
m
b
o
 
=
 
e
r
[
'
c
o
m
b
o
'
]


 
 
 
 
#
 
O
R
 
l
o
g
i
c
 
o
n
 
v
a
l
 
d
a
t
a

 
 
 
 
e
n
t
_
v
a
l
 
=
 
s
t
r
a
t
e
g
i
e
s
[
c
o
m
b
o
[
0
]
]
[
'
e
n
t
r
i
e
s
'
]
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)

 
 
 
 
e
x
t
_
v
a
l
 
=
 
s
t
r
a
t
e
g
i
e
s
[
c
o
m
b
o
[
0
]
]
[
'
e
x
i
t
s
'
]
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]
.
c
o
p
y
(
)

 
 
 
 
f
o
r
 
i
 
i
n
 
c
o
m
b
o
[
1
:
]
:

 
 
 
 
 
 
 
 
e
n
t
_
v
a
l
 
=
 
e
n
t
_
v
a
l
 
|
 
s
t
r
a
t
e
g
i
e
s
[
i
]
[
'
e
n
t
r
i
e
s
'
]
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]

 
 
 
 
 
 
 
 
e
x
t
_
v
a
l
 
=
 
e
x
t
_
v
a
l
 
|
 
s
t
r
a
t
e
g
i
e
s
[
i
]
[
'
e
x
i
t
s
'
]
.
i
l
o
c
[
s
p
l
i
t
_
i
d
x
:
]


 
 
 
 
#
 
G
e
t
 
p
o
s
i
t
i
o
n
 
s
e
r
i
e
s
 
a
n
d
 
O
O
S
 
m
e
t
r
i
c
s

 
 
 
 
p
f
_
v
a
l
 
=
 
r
u
n
_
b
a
c
k
t
e
s
t
(
e
n
t
_
v
a
l
,
 
e
x
t
_
v
a
l
,
 
c
l
o
s
e
_
v
a
l
)

 
 
 
 
m
_
v
a
l
 
=
 
g
e
t
_
m
e
t
r
i
c
s
(
p
f
_
v
a
l
)

 
 
 
 
p
o
s
i
t
i
o
n
_
o
o
s
 
=
 
g
e
t
_
p
o
s
i
t
i
o
n
_
s
e
r
i
e
s
(
e
n
t
_
v
a
l
,
 
e
x
t
_
v
a
l
,
 
c
l
o
s
e
_
v
a
l
)


 
 
 
 
#
 
R
u
n
 
c
o
r
r
e
c
t
e
d
 
B
o
r
u
t
a

 
 
 
 
b
o
r
u
t
a
_
s
c
o
r
e
,
 
r
e
a
l
_
s
h
a
r
p
e
,
 
m
e
d
i
a
n
_
s
h
a
d
o
w
 
=
 
b
o
r
u
t
a
_
v
a
l
i
d
a
t
e
(

 
 
 
 
 
 
 
 
p
o
s
i
t
i
o
n
_
o
o
s
,
 
a
s
s
e
t
_
r
e
t
u
r
n
s
_
o
o
s
,
 
n
_
s
h
u
f
f
l
e
s
=
N
_
S
H
U
F
F
L
E
S

 
 
 
 
)


 
 
 
 
#
 
p
-
v
a
l
u
e
 
=
 
1
 
-
 
b
o
r
u
t
a
_
s
c
o
r
e
 
(
f
r
a
c
t
i
o
n
 
o
f
 
t
i
m
e
s
 
r
e
a
l
 
<
=
 
s
h
a
d
o
w
)

 
 
 
 
p
_
v
a
l
u
e
 
=
 
1
.
0
 
-
 
b
o
r
u
t
a
_
s
c
o
r
e


 
 
 
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
.
a
p
p
e
n
d
(
{

 
 
 
 
 
 
 
 
'
r
a
n
k
'
:
 
i
d
x
 
+
 
1
,

 
 
 
 
 
 
 
 
'
n
a
m
e
'
:
 
e
r
[
'
n
a
m
e
'
]
,

 
 
 
 
 
 
 
 
'
c
o
m
b
o
'
:
 
c
o
m
b
o
,

 
 
 
 
 
 
 
 
'
n
_
s
t
r
a
t
e
g
i
e
s
'
:
 
e
r
[
'
n
_
s
t
r
a
t
e
g
i
e
s
'
]
,

 
 
 
 
 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
e
r
[
'
i
s
_
s
h
a
r
p
e
'
]
,

 
 
 
 
 
 
 
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
r
e
a
l
_
s
h
a
r
p
e
,

 
 
 
 
 
 
 
 
'
o
o
s
_
r
e
t
u
r
n
'
:
 
m
_
v
a
l
[
'
t
o
t
a
l
_
r
e
t
u
r
n
'
]
,

 
 
 
 
 
 
 
 
'
o
o
s
_
m
a
x
d
d
'
:
 
m
_
v
a
l
[
'
m
a
x
_
d
d
'
]
,

 
 
 
 
 
 
 
 
'
o
o
s
_
t
r
a
d
e
s
'
:
 
m
_
v
a
l
[
'
n
_
t
r
a
d
e
s
'
]
,

 
 
 
 
 
 
 
 
'
b
o
r
u
t
a
_
s
c
o
r
e
'
:
 
b
o
r
u
t
a
_
s
c
o
r
e
,

 
 
 
 
 
 
 
 
'
p
_
v
a
l
u
e
'
:
 
p
_
v
a
l
u
e
,

 
 
 
 
 
 
 
 
'
m
e
d
i
a
n
_
s
h
a
d
o
w
'
:
 
m
e
d
i
a
n
_
s
h
a
d
o
w
,

 
 
 
 
}
)


 
 
 
 
i
f
 
(
i
d
x
 
+
 
1
)
 
%
 
5
 
=
=
 
0
:

 
 
 
 
 
 
 
 
p
r
i
n
t
(
f
"
 
 
C
o
m
p
l
e
t
e
d
 
{
i
d
x
 
+
 
1
}
/
{
t
o
p
_
n
}
.
.
.
"
)


#
 
A
p
p
l
y
 
B
e
n
j
a
m
i
n
i
-
H
o
c
h
b
e
r
g
 
c
o
r
r
e
c
t
i
o
n

r
a
w
_
p
v
a
l
s
 
=
 
[
b
r
[
'
p
_
v
a
l
u
e
'
]
 
f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
]

b
h
_
r
e
j
e
c
t
e
d
,
 
b
h
_
a
d
j
u
s
t
e
d
 
=
 
b
e
n
j
a
m
i
n
i
_
h
o
c
h
b
e
r
g
(
r
a
w
_
p
v
a
l
s
,
 
f
d
r
=
0
.
0
5
)


f
o
r
 
i
,
 
b
r
 
i
n
 
e
n
u
m
e
r
a
t
e
(
b
o
r
u
t
a
_
r
e
s
u
l
t
s
)
:

 
 
 
 
b
r
[
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
]
 
=
 
b
h
_
a
d
j
u
s
t
e
d
[
i
]

 
 
 
 
b
r
[
'
v
e
r
d
i
c
t
'
]
 
=
 
'
C
O
N
F
I
R
M
E
D
'
 
i
f
 
b
h
_
r
e
j
e
c
t
e
d
[
i
]
 
e
l
s
e
 
'
R
E
J
E
C
T
E
D
'


#
 
P
r
i
n
t
 
r
e
s
u
l
t
s

p
r
i
n
t
(
)

p
r
i
n
t
(
f
"
{
'
R
a
n
k
'
:
<
5
}
 
{
'
E
n
s
e
m
b
l
e
'
:
<
4
0
}
 
{
'
I
S
 
S
h
r
p
'
:
>
8
}
 
{
'
O
O
S
 
S
h
r
p
'
:
>
9
}
 
{
'
S
c
o
r
e
'
:
>
7
}
 
{
'
R
a
w
 
p
'
:
>
7
}
 
{
'
A
d
j
 
p
'
:
>
7
}
 
{
'
V
e
r
d
i
c
t
'
:
>
1
2
}
"
)

p
r
i
n
t
(
"
=
"
 
*
 
1
0
0
)

f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
:

 
 
 
 
i
s
_
s
 
=
 
f
"
{
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
:
.
3
f
}
"
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
'
N
/
A
'

 
 
 
 
o
o
s
_
s
 
=
 
f
"
{
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
:
.
3
f
}
"
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
'
N
/
A
'

 
 
 
 
b
_
s
 
=
 
f
"
{
b
r
[
'
b
o
r
u
t
a
_
s
c
o
r
e
'
]
:
.
1
%
}
"

 
 
 
 
r
a
w
_
p
 
=
 
f
"
{
b
r
[
'
p
_
v
a
l
u
e
'
]
:
.
3
f
}
"

 
 
 
 
a
d
j
_
p
 
=
 
f
"
{
b
r
[
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
]
:
.
3
f
}
"

 
 
 
 
i
c
o
n
 
=
 
'
C
O
N
F
I
R
M
E
D
'
 
i
f
 
b
r
[
'
v
e
r
d
i
c
t
'
]
 
=
=
 
'
C
O
N
F
I
R
M
E
D
'
 
e
l
s
e
 
'
R
E
J
E
C
T
E
D
'

 
 
 
 
p
r
i
n
t
(
f
"
{
b
r
[
'
r
a
n
k
'
]
:
<
5
}
 
{
b
r
[
'
n
a
m
e
'
]
:
<
4
0
}
 
{
i
s
_
s
:
>
8
}
 
{
o
o
s
_
s
:
>
9
}
 
{
b
_
s
:
>
7
}
 
{
r
a
w
_
p
:
>
7
}
 
{
a
d
j
_
p
:
>
7
}
 
{
i
c
o
n
:
>
1
2
}
"
)


n
_
c
o
n
f
i
r
m
e
d
 
=
 
s
u
m
(
1
 
f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
 
i
f
 
b
r
[
'
v
e
r
d
i
c
t
'
]
 
=
=
 
'
C
O
N
F
I
R
M
E
D
'
)

n
_
r
e
j
e
c
t
e
d
 
=
 
s
u
m
(
1
 
f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
 
i
f
 
b
r
[
'
v
e
r
d
i
c
t
'
]
 
=
=
 
'
R
E
J
E
C
T
E
D
'
)

p
r
i
n
t
(
f
"
\
n
B
H
-
C
o
n
f
i
r
m
e
d
:
 
{
n
_
c
o
n
f
i
r
m
e
d
}
 
/
 
{
t
o
p
_
n
}
 
 
|
 
 
R
e
j
e
c
t
e
d
:
 
{
n
_
r
e
j
e
c
t
e
d
}
 
/
 
{
t
o
p
_
n
}
"
)

i
f
 
n
_
c
o
n
f
i
r
m
e
d
 
=
=
 
0
:

 
 
 
 
p
r
i
n
t
(
"
(
I
f
 
0
 
c
o
n
f
i
r
m
e
d
:
 
t
h
e
 
s
t
r
a
t
e
g
i
e
s
 
m
a
y
 
n
o
t
 
h
a
v
e
 
r
e
a
l
 
e
d
g
e
 
o
n
 
t
h
i
s
 
t
i
c
k
e
r
'
s
 
O
O
S
 
p
e
r
i
o
d
)
"
)

## Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 20))
fig.suptitle(f'Boruta Ensemble Validation - {TICKER}', fontsize=16, fontweight='bold', y=0.98)

# --- Subplot 1: IS vs OOS Sharpe bar chart ---
ax1 = axes[0]
n_bars = len(boruta_results)
x = np.arange(n_bars)
width = 0.35

is_sharpes = [br['is_sharpe'] if pd.notna(br['is_sharpe']) else 0 for br in boruta_results]
oos_sharpes = [br['oos_sharpe'] if pd.notna(br['oos_sharpe']) else 0 for br in boruta_results]
colors_is = ['#2ecc71' if br['verdict'] == 'CONFIRMED' else '#e74c3c' for br in boruta_results]
colors_oos = ['#27ae60' if br['verdict'] == 'CONFIRMED' else '#c0392b' for br in boruta_results]

ax1.bar(x - width/2, is_sharpes, width, color=colors_is, alpha=0.8, label='IS Sharpe')
ax1.bar(x + width/2, oos_sharpes, width, color=colors_oos, alpha=0.6, label='OOS Sharpe')
ax1.set_xlabel('Ensemble Rank')
ax1.set_ylabel('Sharpe Ratio')
ax1.set_title('Top 25 Ensembles: IS vs OOS Sharpe (Green=Confirmed, Red=Rejected)')
ax1.set_xticks(x)
ax1.set_xticklabels([str(br['rank']) for br in boruta_results], fontsize=8)
ax1.legend()
ax1.axhline(y=0, color='black', linewidth=0.5)

# --- Subplot 2: Strategy inclusion heatmap ---
ax2 = axes[1]
confirmed_results = [br for br in boruta_results if br['verdict'] == 'CONFIRMED']

# Count strategy appearances
inclusion_data = {}
for sname in strategy_names:
    in_top25 = sum(1 for br in boruta_results if sname in br['name'])
    in_confirmed = sum(1 for br in confirmed_results if sname in br['name'])
    inclusion_data[sname] = {'In Top 25': in_top25, 'In Confirmed': in_confirmed}

df_inclusion = pd.DataFrame(inclusion_data).T
if len(df_inclusion) > 0 and df_inclusion.values.max() > 0:
    sns.heatmap(df_inclusion, annot=True, fmt='d', cmap='YlOrRd', ax=ax2,
                linewidths=0.5, cbar_kws={'label': 'Count'})
    ax2.set_title('Strategy Inclusion Frequency in Ensembles')
    ax2.set_ylabel('Strategy')
else:
    ax2.text(0.5, 0.5, 'No confirmed ensembles to display',
             ha='center', va='center', transform=ax2.transAxes, fontsize=14)
    ax2.set_title('Strategy Inclusion Frequency in Ensembles')

# --- Subplot 3: Equity curves for top 3 confirmed ensembles + buy-and-hold ---
ax3 = axes[2]

# Buy and hold on full data
pf_bh = vbt.Portfolio.from_holding(close_full, init_cash=INIT_CASH, fees=FEES, freq='D')
equity_bh = pf_bh.value()
ax3.plot(equity_bh.index, equity_bh.values, label='Buy & Hold', color='gray', linewidth=1.5, alpha=0.7)

# Top 3 confirmed
top3_confirmed = confirmed_results[:3]
colors_eq = ['#2ecc71', '#3498db', '#9b59b6']

for i, br in enumerate(top3_confirmed):
    combo = br['combo']
    # Full-sample signals
    ent_full = strategies[combo[0]]['entries'].copy()
    ext_full = strategies[combo[0]]['exits'].copy()
    for ci in combo[1:]:
        ent_full = ent_full | strategies[ci]['entries']
        ext_full = ext_full | strategies[ci]['exits']

    pf_full = run_backtest(ent_full, ext_full, close_full)
    equity = pf_full.value()
    short_name = br['name'] if len(br['name']) < 40 else br['name'][:37] + '...'
    ax3.plot(equity.index, equity.values, label=f"#{br['rank']} {short_name}",
             color=colors_eq[i], linewidth=1.5)

# Mark train/val split
split_date = close.index[split_idx]
ax3.axvline(x=split_date, color='red', linestyle='--', alpha=0.7, label='Train/Val Split')

ax3.set_title('Equity Curves: Top 3 Confirmed Ensembles vs Buy & Hold')
ax3.set_xlabel('Date')
ax3.set_ylabel('Portfolio Value ($)')
ax3.legend(fontsize=8, loc='upper left')
ax3.set_xlim(close.index[0], close.index[-1])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Summary & Recommendations

In [ ]:
print("=" * 80)
print(f"  BORUTA ENSEMBLE VALIDATION SUMMARY - {TICKER}")
print("=" * 80)

# Individual strategy analysis
print("\n--- Individual Strategy Performance ---")
for ir in individual_results:
    is_s = f"{ir['is_sharpe']:.3f}" if pd.notna(ir['is_sharpe']) else 'N/A'
    oos_s = f"{ir['oos_sharpe']:.3f}" if pd.notna(ir['oos_sharpe']) else 'N/A'
    status = 'GOOD' if (pd.notna(ir['is_sharpe']) and ir['is_sharpe'] > 0 and
                        pd.notna(ir['oos_sharpe']) and ir['oos_sharpe'] > 0) else 'WEAK'
    print(f"  {ir['name']:20s}  IS={is_s:>7s}  OOS={oos_s:>7s}  [{status}]")

# Confirmed ensembles
print(f"\n--- Boruta-Confirmed Ensembles ({n_confirmed} of {top_n}) ---")
if confirmed_results:
    for br in confirmed_results:
        is_s = f"{br['is_sharpe']:.3f}" if pd.notna(br['is_sharpe']) else 'N/A'
        oos_s = f"{br['oos_sharpe']:.3f}" if pd.notna(br['oos_sharpe']) else 'N/A'
        print(f"  Rank {br['rank']:>2d}: {br['name']}")
        print(f"           IS Sharpe={is_s}  OOS Sharpe={oos_s}  Boruta={br['boruta_score']:.1%}")
else:
    print("  No ensembles passed Boruta validation.")
    print("  Consider: adjusting parameters, adding more strategies, or using a different ticker.")

# Recommendation
print("\n--- Recommendation ---")
if confirmed_results:
    best = confirmed_results[0]
    print(f"  Best Ensemble: {best['name']}")
    print(f"  IS Sharpe:  {best['is_sharpe']:.3f}" if pd.notna(best['is_sharpe']) else "  IS Sharpe:  N/A")
    print(f"  OOS Sharpe: {best['oos_sharpe']:.3f}" if pd.notna(best['oos_sharpe']) else "  OOS Sharpe: N/A")
    print(f"  Boruta:     {best['boruta_score']:.1%}")
    print(f"  Components: {[strategies[i]['name'] for i in best['combo']]}")
else:
    print("  No recommendation - no ensembles passed validation.")

# Correlation matrix between confirmed ensemble returns
if len(confirmed_results) >= 2:
    print("\n--- Return Correlation Matrix (Confirmed Ensembles, OOS) ---")
    corr_returns = {}
    for br in confirmed_results:
        combo = br['combo']
        ent_val = strategies[combo[0]]['entries'].iloc[split_idx:].copy()
        ext_val = strategies[combo[0]]['exits'].iloc[split_idx:].copy()
        for ci in combo[1:]:
            ent_val = ent_val | strategies[ci]['entries'].iloc[split_idx:]
            ext_val = ext_val | strategies[ci]['exits'].iloc[split_idx:]
        pf_val = run_backtest(ent_val, ext_val, close_val)
        short_name = br['name'] if len(br['name']) < 25 else br['name'][:22] + '...'
        corr_returns[short_name] = pf_val.daily_returns()

    df_corr = pd.DataFrame(corr_returns)
    print(df_corr.corr().round(3).to_string())

print("\n" + "=" * 80)

## Export Results

In [ ]:
#
 
P
r
e
p
a
r
e
 
e
x
p
o
r
t
 
d
a
t
a

c
o
n
f
i
r
m
e
d
_
r
e
s
u
l
t
s
 
=
 
[
b
r
 
f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s
 
i
f
 
b
r
[
'
v
e
r
d
i
c
t
'
]
 
=
=
 
'
C
O
N
F
I
R
M
E
D
'
]

b
e
s
t
_
c
o
n
f
i
r
m
e
d
_
n
a
m
e
 
=
 
c
o
n
f
i
r
m
e
d
_
r
e
s
u
l
t
s
[
0
]
[
'
n
a
m
e
'
]
 
i
f
 
c
o
n
f
i
r
m
e
d
_
r
e
s
u
l
t
s
 
e
l
s
e
 
'
N
o
n
e
'


e
x
p
o
r
t
 
=
 
{

 
 
 
 
'
t
i
c
k
e
r
'
:
 
T
I
C
K
E
R
,

 
 
 
 
'
d
a
t
e
_
r
u
n
'
:
 
s
t
r
(
p
d
.
T
i
m
e
s
t
a
m
p
.
n
o
w
(
)
)
,

 
 
 
 
'
t
r
a
i
n
_
p
e
r
i
o
d
'
:
 
f
"
{
c
l
o
s
e
_
t
r
a
i
n
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
t
o
 
{
c
l
o
s
e
_
t
r
a
i
n
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
"
,

 
 
 
 
'
v
a
l
_
p
e
r
i
o
d
'
:
 
f
"
{
c
l
o
s
e
_
v
a
l
.
i
n
d
e
x
[
0
]
.
d
a
t
e
(
)
}
 
t
o
 
{
c
l
o
s
e
_
v
a
l
.
i
n
d
e
x
[
-
1
]
.
d
a
t
e
(
)
}
"
,

 
 
 
 
'
n
_
s
h
u
f
f
l
e
s
'
:
 
N
_
S
H
U
F
F
L
E
S
,

 
 
 
 
'
b
o
r
u
t
a
_
m
e
t
h
o
d
'
:
 
'
s
h
u
f
f
l
e
_
a
s
s
e
t
_
r
e
t
u
r
n
s
'
,

 
 
 
 
'
m
u
l
t
i
p
l
e
_
t
e
s
t
i
n
g
_
c
o
r
r
e
c
t
i
o
n
'
:
 
'
B
e
n
j
a
m
i
n
i
-
H
o
c
h
b
e
r
g
 
F
D
R
 
@
 
5
%
'
,

 
 
 
 
'
i
n
d
i
v
i
d
u
a
l
_
r
e
s
u
l
t
s
'
:
 
[

 
 
 
 
 
 
 
 
{

 
 
 
 
 
 
 
 
 
 
 
 
'
n
a
m
e
'
:
 
i
r
[
'
n
a
m
e
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
i
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
r
e
t
u
r
n
'
:
 
f
l
o
a
t
(
i
r
[
'
i
s
_
r
e
t
u
r
n
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
i
s
_
r
e
t
u
r
n
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
m
a
x
d
d
'
:
 
f
l
o
a
t
(
i
r
[
'
i
s
_
m
a
x
d
d
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
i
s
_
m
a
x
d
d
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
t
r
a
d
e
s
'
:
 
i
n
t
(
i
r
[
'
i
s
_
t
r
a
d
e
s
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
i
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
r
e
t
u
r
n
'
:
 
f
l
o
a
t
(
i
r
[
'
o
o
s
_
r
e
t
u
r
n
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
o
o
s
_
r
e
t
u
r
n
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
m
a
x
d
d
'
:
 
f
l
o
a
t
(
i
r
[
'
o
o
s
_
m
a
x
d
d
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
i
r
[
'
o
o
s
_
m
a
x
d
d
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
t
r
a
d
e
s
'
:
 
i
n
t
(
i
r
[
'
o
o
s
_
t
r
a
d
e
s
'
]
)
,

 
 
 
 
 
 
 
 
}

 
 
 
 
 
 
 
 
f
o
r
 
i
r
 
i
n
 
i
n
d
i
v
i
d
u
a
l
_
r
e
s
u
l
t
s

 
 
 
 
]
,

 
 
 
 
'
t
o
p
_
e
n
s
e
m
b
l
e
s
'
:
 
[

 
 
 
 
 
 
 
 
{

 
 
 
 
 
 
 
 
 
 
 
 
'
r
a
n
k
'
:
 
b
r
[
'
r
a
n
k
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
n
a
m
e
'
:
 
b
r
[
'
n
a
m
e
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
n
_
s
t
r
a
t
e
g
i
e
s
'
:
 
b
r
[
'
n
_
s
t
r
a
t
e
g
i
e
s
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
b
o
r
u
t
a
_
s
c
o
r
e
'
:
 
f
l
o
a
t
(
b
r
[
'
b
o
r
u
t
a
_
s
c
o
r
e
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
p
_
v
a
l
u
e
'
:
 
f
l
o
a
t
(
b
r
[
'
p
_
v
a
l
u
e
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
:
 
f
l
o
a
t
(
b
r
[
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
v
e
r
d
i
c
t
'
:
 
b
r
[
'
v
e
r
d
i
c
t
'
]
,

 
 
 
 
 
 
 
 
}

 
 
 
 
 
 
 
 
f
o
r
 
b
r
 
i
n
 
b
o
r
u
t
a
_
r
e
s
u
l
t
s

 
 
 
 
]
,

 
 
 
 
'
c
o
n
f
i
r
m
e
d
_
e
n
s
e
m
b
l
e
s
'
:
 
[

 
 
 
 
 
 
 
 
{

 
 
 
 
 
 
 
 
 
 
 
 
'
r
a
n
k
'
:
 
b
r
[
'
r
a
n
k
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
n
a
m
e
'
:
 
b
r
[
'
n
a
m
e
'
]
,

 
 
 
 
 
 
 
 
 
 
 
 
'
i
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
i
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
o
o
s
_
s
h
a
r
p
e
'
:
 
f
l
o
a
t
(
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
i
f
 
p
d
.
n
o
t
n
a
(
b
r
[
'
o
o
s
_
s
h
a
r
p
e
'
]
)
 
e
l
s
e
 
N
o
n
e
,

 
 
 
 
 
 
 
 
 
 
 
 
'
b
o
r
u
t
a
_
s
c
o
r
e
'
:
 
f
l
o
a
t
(
b
r
[
'
b
o
r
u
t
a
_
s
c
o
r
e
'
]
)
,

 
 
 
 
 
 
 
 
 
 
 
 
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
:
 
f
l
o
a
t
(
b
r
[
'
b
h
_
a
d
j
u
s
t
e
d
_
p
'
]
)
,

 
 
 
 
 
 
 
 
}

 
 
 
 
 
 
 
 
f
o
r
 
b
r
 
i
n
 
c
o
n
f
i
r
m
e
d
_
r
e
s
u
l
t
s

 
 
 
 
]
,

 
 
 
 
'
r
e
c
o
m
m
e
n
d
a
t
i
o
n
'
:
 
b
e
s
t
_
c
o
n
f
i
r
m
e
d
_
n
a
m
e

}


#
 
S
a
v
e
 
J
S
O
N

e
x
p
o
r
t
_
d
i
r
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
o
s
.
p
a
t
h
.
d
i
r
n
a
m
e
(
o
s
.
p
a
t
h
.
a
b
s
p
a
t
h
(
'
.
'
)
)
,
 
'
e
x
p
o
r
t
s
'
)

o
s
.
m
a
k
e
d
i
r
s
(
e
x
p
o
r
t
_
d
i
r
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)


j
s
o
n
_
p
a
t
h
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
e
x
p
o
r
t
_
d
i
r
,
 
f
'
b
o
r
u
t
a
_
e
n
s
e
m
b
l
e
_
{
T
I
C
K
E
R
}
.
j
s
o
n
'
)

w
i
t
h
 
o
p
e
n
(
j
s
o
n
_
p
a
t
h
,
 
'
w
'
)
 
a
s
 
f
:

 
 
 
 
j
s
o
n
.
d
u
m
p
(
e
x
p
o
r
t
,
 
f
,
 
i
n
d
e
n
t
=
2
)

p
r
i
n
t
(
f
"
J
S
O
N
 
s
a
v
e
d
 
t
o
:
 
{
j
s
o
n
_
p
a
t
h
}
"
)


#
 
S
a
v
e
 
C
S
V
 
o
f
 
t
o
p
 
e
n
s
e
m
b
l
e
s

d
f
_
e
x
p
o
r
t
 
=
 
p
d
.
D
a
t
a
F
r
a
m
e
(
b
o
r
u
t
a
_
r
e
s
u
l
t
s
)

c
s
v
_
p
a
t
h
 
=
 
o
s
.
p
a
t
h
.
j
o
i
n
(
e
x
p
o
r
t
_
d
i
r
,
 
f
'
b
o
r
u
t
a
_
e
n
s
e
m
b
l
e
_
{
T
I
C
K
E
R
}
.
c
s
v
'
)

d
f
_
e
x
p
o
r
t
.
t
o
_
c
s
v
(
c
s
v
_
p
a
t
h
,
 
i
n
d
e
x
=
F
a
l
s
e
)

p
r
i
n
t
(
f
"
C
S
V
 
s
a
v
e
d
 
t
o
:
 
 
{
c
s
v
_
p
a
t
h
}
"
)


p
r
i
n
t
(
"
\
n
D
o
n
e
!
"
)